# 03. CLAP vs M2D-CLAP — Audio-Text Matching Comparison

We compare two contrastive audio-language models on the same dataset (ESC-50, 2,000 clips, 50 balanced classes):

| | CLAP (LAION-AI 2023) | M2D-CLAP$_{2025}$ (Niizumi et al., IEEE Access 2025) |
|---|---|---|
| audio encoder  | HTSAT-tiny | M2D ViT-Base (jointly trained with masked modeling) |
| text encoder   | RoBERTa-base | BERT-base (Stage-2 fine-tuned on AudioCaps + Clotho) |
| training data  | LAION-Audio-630k | AudioSet + VGGSound + WavCaps + AudioCaps + Clotho |
| released emb dim | 512 | 768 |

Both notebook 01 and notebook 02 used the **same** captions/prompts/audio set, so the only thing that changes here is the model.

**Why we don't run plain N-vs-N retrieval on ESC-50.** ESC-50 is a *classification* dataset: each class has 40 audios with the same caption (`'This is a sound of dog.'`). In a 2000×2000 retrieval matrix, the 40 in-class captions are duplicates of each other, which (a) inflates ranks under tie-breaking quirks and (b) penalizes a model that is otherwise semantically correct. The fair setups on ESC-50 are:

1. **Zero-shot classification**: each audio is matched against the 50 *unique* class prompts → top-1 / top-5 accuracy.
2. **A→prompts retrieval**: same as (1) reported as R@1 / R@5 / R@10.
3. **prompts→A retrieval**: each of the 50 class prompts is a query against the 2000 audios; we report R@K against the multi-positive set of 40 in-class audios per query.

We also visualize the cosine-similarity geometry, splitting *negatives* into cross-class pairs (true negatives) vs same-class duplicate captions (which shouldn't count as negatives).

In [ ]:
import os, sys
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
if str(PROJECT_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / 'src'))

RESULTS = PROJECT_DIR / 'results'
FIGS    = RESULTS / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(context='notebook', style='whitegrid')

from analyze import (
    load_bundle, l2_normalize, zero_shot_accuracy,
    class_prompt_retrieval, class_prompt_retrieval_text2audio,
)

## 1. Load both bundles

In [ ]:
clap = load_bundle('CLAP',     RESULTS / 'clap')
m2d  = load_bundle('M2D-CLAP', RESULTS / 'm2d_clap')

for b in [clap, m2d]:
    print(f"{b['name']:<10}  audio {b['audio_emb'].shape}  text {b['text_emb'].shape}")

assert clap['audio_emb'].shape[0] == m2d['audio_emb'].shape[0]
assert clap['captions'] == m2d['captions'], 'captions must match for fair comparison'
N = clap['audio_emb'].shape[0]
print('aligned samples:', N)

## 2. Retrieval against the 50 unique class prompts

**A→prompts**: each of 2000 audios finds its class among 50 prompts.
**prompts→A**: each of 50 class prompts finds an in-class audio among 2000 (multi-positive).

On ESC-50 these are the only retrieval setups that are not corrupted by caption duplication.

In [ ]:
def labels_to_targets(b):
    rd = RESULTS / ('clap' if b['name']=='CLAP' else 'm2d_clap')
    class_names = (rd / 'class_names.txt').read_text().splitlines()
    cemb = np.load(rd / 'class_text_embeddings.npy')
    name2idx = {n:i for i,n in enumerate(class_names)}
    targets = np.array([name2idx[c] for c in b['labels']])
    return targets, cemb, class_names

rows = []
for b in [clap, m2d]:
    targets, cemb, _ = labels_to_targets(b)
    a2t = class_prompt_retrieval(b['audio_emb'], cemb, targets)
    t2a = class_prompt_retrieval_text2audio(b['audio_emb'], cemb, targets)
    for direction, m in [('a2t', a2t), ('t2a', t2a)]:
        rows.append({'model': b['name'], 'direction': direction, **m,
                     'paired_sim_mean': float(np.mean(b['paired'])),
                     'paired_sim_std':  float(np.std(b['paired']))})
tbl = pd.DataFrame(rows)
tbl.to_csv(FIGS / 'retrieval_metrics.csv', index=False)
tbl[['model','direction','R@1','R@5','R@10','MedR','MeanR']]\
    .style.format(precision=4)

## 3. Paired similarity geometry

Three histograms per model:
- **paired positives** (red): cos(audio_i, caption_i)
- **in-class positives off-diagonal** (green): cos(audio_i, caption_j) where i≠j but same class
- **cross-class negatives** (blue): cos(audio_i, caption_j) where class differs

A well-aligned model puts red+green to the right and blue to the left.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, b in zip(axes, [clap, m2d]):
    sim = b['sim']
    n = sim.shape[0]
    labels_arr = np.array(b['labels'])
    same_class = labels_arr[:, None] == labels_arr[None, :]
    diag = np.eye(n, dtype=bool)
    cross_class_neg = sim[~same_class]
    in_class_pos    = sim[same_class & ~diag]
    pair_pos        = b['paired']
    ax.hist(cross_class_neg, bins=80, alpha=0.55, density=True,
            color='steelblue', label='cross-class negatives')
    ax.hist(in_class_pos,    bins=40, alpha=0.55, density=True,
            color='seagreen',  label='in-class positives (off-diag)')
    ax.hist(pair_pos,        bins=40, alpha=0.85, density=True,
            color='tomato',    label='paired positives (diag)')
    gap = pair_pos.mean() - cross_class_neg.mean()
    ax.axvline(pair_pos.mean(),        color='tomato',    linestyle='--', linewidth=1)
    ax.axvline(cross_class_neg.mean(), color='steelblue', linestyle='--', linewidth=1)
    ax.set_title(f"{b['name']}    pos μ={pair_pos.mean():.3f}, neg μ={cross_class_neg.mean():.3f}, gap={gap:.3f}")
    ax.set_xlabel('cosine similarity'); ax.legend(fontsize=8)
axes[0].set_ylabel('density')
plt.tight_layout()
plt.savefig(FIGS / 'paired_similarity_distribution.png', dpi=150)
plt.show()

## 4. Retrieval bar chart

In [ ]:
long = tbl.melt(id_vars=['model','direction'], value_vars=['R@1','R@5','R@10'],
                var_name='metric', value_name='score')
g = sns.catplot(
    data=long, kind='bar',
    x='metric', y='score', hue='model', col='direction',
    height=4, aspect=1.1, palette=['#4C72B0', '#DD8452'],
)
g.set_axis_labels('', 'Recall@K')
g.set(ylim=(0, 1))
for ax in g.axes.flat:
    for p in ax.patches:
        h = p.get_height()
        if h <= 0: continue
        ax.annotate(f'{h:.3f}', (p.get_x()+p.get_width()/2, h+0.01), ha='center', fontsize=9)
g.figure.suptitle('Retrieval against 50 ESC-50 class prompts', y=1.04)
g.figure.savefig(FIGS / 'retrieval_bars.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Similarity matrix heatmaps (first K samples)

In [ ]:
K = 50
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
vmin = min(clap['sim'][:K,:K].min(), m2d['sim'][:K,:K].min())
vmax = max(clap['sim'][:K,:K].max(), m2d['sim'][:K,:K].max())
for ax, b in zip(axes, [clap, m2d]):
    im = ax.imshow(b['sim'][:K, :K], cmap='viridis', vmin=vmin, vmax=vmax, aspect='equal')
    ax.set_title(f"{b['name']} similarity matrix (first {K})")
    ax.set_xlabel('text idx'); ax.set_ylabel('audio idx')
fig.colorbar(im, ax=axes, fraction=0.025)
fig.suptitle('Diagonal prominence ↔ better audio-text alignment', y=1.02)
plt.savefig(FIGS / 'similarity_matrix_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Zero-shot ESC-50 classification

Audio → argmax over 50 class prompts. Top-1 / top-5 accuracy.

In [ ]:
def zs_for(b):
    targets, cemb, class_names = labels_to_targets(b)
    return zero_shot_accuracy(b['audio_emb'], cemb, targets), class_names, targets

(zs_clap, class_names_clap, targets_clap) = zs_for(clap)
(zs_m2d , class_names_m2d , targets_m2d ) = zs_for(m2d)

zs_table = pd.DataFrame({
    'model'    : ['CLAP', 'M2D-CLAP'],
    'top1_acc' : [zs_clap['top1'], zs_m2d['top1']],
    'top5_acc' : [zs_clap['top5'], zs_m2d['top5']],
})
zs_table.to_csv(FIGS / 'zero_shot_accuracy.csv', index=False)
zs_table.style.format(precision=4)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
x = np.arange(2)
ax.bar(x-0.18, zs_table['top1_acc'], width=0.35, label='top-1', color='#4C72B0')
ax.bar(x+0.18, zs_table['top5_acc'], width=0.35, label='top-5', color='#DD8452')
ax.set_xticks(x); ax.set_xticklabels(zs_table['model'])
ax.set_ylim(0, 1); ax.set_ylabel('accuracy')
ax.set_title('ESC-50 zero-shot classification')
for i,(t1,t5) in enumerate(zip(zs_table['top1_acc'], zs_table['top5_acc'])):
    ax.annotate(f'{t1:.3f}', (i-0.18, t1+0.01), ha='center', fontsize=9)
    ax.annotate(f'{t5:.3f}', (i+0.18, t5+0.01), ha='center', fontsize=9)
ax.legend()
plt.tight_layout(); plt.savefig(FIGS / 'zero_shot_bars.png', dpi=150)
plt.show()

## 7. Per-class top-1 accuracy: where do they disagree?

In [ ]:
def per_class_top1(b):
    targets, cemb, class_names = labels_to_targets(b)
    logits = l2_normalize(b['audio_emb']) @ l2_normalize(cemb).T
    pred = logits.argmax(axis=1)
    acc = pd.DataFrame({'class': [class_names[t] for t in targets], 'correct': (pred==targets).astype(int)})
    return acc.groupby('class')['correct'].mean()

pc_clap = per_class_top1(clap).rename('CLAP')
pc_m2d  = per_class_top1(m2d ).rename('M2D-CLAP')
pc = pd.concat([pc_clap, pc_m2d], axis=1).fillna(0)
pc['delta'] = pc['M2D-CLAP'] - pc['CLAP']
pc_sorted = pc.sort_values('delta', ascending=False)
print('Top 10 classes where M2D-CLAP gains most over CLAP:')
print(pc_sorted.head(10).to_string(float_format=lambda v: f'{v:.3f}'))
print('\nTop 10 classes where CLAP still wins (negative delta):')
print(pc_sorted.tail(10).iloc[::-1].to_string(float_format=lambda v: f'{v:.3f}'))
pc_sorted.to_csv(FIGS / 'per_class_top1.csv')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))
pc_sorted[['CLAP','M2D-CLAP']].plot.barh(ax=ax, color=['#4C72B0', '#DD8452'])
ax.set_title('Per-class zero-shot top-1 accuracy (sorted by M2D-CLAP gain)')
ax.set_xlim(0, 1); ax.invert_yaxis(); ax.set_xlabel('top-1 accuracy')
plt.tight_layout(); plt.savefig(FIGS / 'per_class_top1.png', dpi=150)
plt.show()

## 8. Qualitative: nearest-prompt for a few sample audios

For 4 random clips we list the top-3 *class prompts* CLAP and M2D-CLAP each retrieve. ★ marks the ground-truth class.

In [ ]:
def topk_prompts(b, audio_idx, k=3):
    targets, cemb, class_names = labels_to_targets(b)
    a = l2_normalize(b['audio_emb'][audio_idx:audio_idx+1])
    c = l2_normalize(cemb)
    scores = (a @ c.T).ravel()
    order = np.argsort(-scores)[:k]
    gt = b['labels'][audio_idx]
    return [(class_names[j], float(scores[j]), class_names[j] == gt) for j in order]

rng = np.random.default_rng(0)
examples = rng.choice(N, size=4, replace=False)
for i in examples:
    print('=' * 72)
    print(f"audio idx {i:>4}  ground truth: {clap['labels'][i]!r}")
    print('-- CLAP top-3 --')
    for cls, s, ok in topk_prompts(clap, i):
        marker = '★' if ok else ' '
        print(f'  {marker} {cls:<25} sim={s:+.3f}')
    print('-- M2D-CLAP top-3 --')
    for cls, s, ok in topk_prompts(m2d, i):
        marker = '★' if ok else ' '
        print(f'  {marker} {cls:<25} sim={s:+.3f}')

## 9. Summary card

In [ ]:
summary_lines = [f'Dataset: ESC-50, N = {N}, classes = 50']
for name in ['CLAP', 'M2D-CLAP']:
    sub = tbl[tbl['model']==name]
    a2t = sub[sub['direction']=='a2t'].iloc[0]
    t2a = sub[sub['direction']=='t2a'].iloc[0]
    summary_lines.append(
        f"{name:<9} | A→prompts R@1={a2t['R@1']:.3f}  R@5={a2t['R@5']:.3f}  | "
        f"prompts→A R@1={t2a['R@1']:.3f}  R@5={t2a['R@5']:.3f}"
    )
summary_lines.append(
    f"Zero-shot top-1: CLAP={zs_clap['top1']:.4f} | M2D-CLAP={zs_m2d['top1']:.4f}"
)
summary = '\n'.join(summary_lines)
print(summary)
(FIGS / 'summary.txt').write_text(summary)